# Exemplos do Capítulo 3: Modelos de Redes Estáticas

Este notebook contém os exemplos práticos de código discutidos no capítulo 3. Você encontrará exemplos sobre Teoria dos Grafos, Análise de Rede com NetworkX, Integração com APIs Biológicas e Simulações de Controle Metabólico.

## Instalação de Dependências

Execute o comando abaixo para instalar as bibliotecas citadas neste capítulo.

In [ ]:
!pip install networkx matplotlib numpy scipy requests

## 1. Criando e Analisando Grafos com NetworkX

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Criar grafo nao-direcionado
G = nx.Graph()

# Adicionar nos e arestas
G.add_nodes_from([1, 2, 3, 4, 5])
G.add_edges_from([(1,2), (1,3), (2,3), (3,4), (4,5)])

# Propriedades basicas
print(f"Numero de nos: {G.number_of_nodes()}")
print(f"Numero de arestas: {G.number_of_edges()}")

In [ ]:
# Calcular distancias
print(f"Diametro: {nx.diameter(G)}")
print(f"Caminho medio: {nx.average_shortest_path_length(G)}")

# Visualizar
nx.draw(G, with_labels=True, node_color='lightblue',
        node_size=500, font_size=16)
plt.show()

## 2. Análise de Matriz de Adjacência

In [ ]:
import numpy as np
import networkx as nx

# Criar grafo de exemplo
G = nx.Graph()
G.add_edges_from([(1,2), (1,3), (2,3), (3,4)])

# Obter matriz de adjacencia
A = nx.adjacency_matrix(G).todense()
print("Matriz de Adjacencia:")
print(A)

# Calcular graus a partir da matriz
degrees = np.sum(A, axis=1)
print(f"\nGraus dos vertices: {degrees.T}")

In [ ]:
# Matriz de adjacencia elevada ao quadrado
A2 = np.linalg.matrix_power(A, 2)
print("\nMatriz A^2 (caminhos de comprimento 2):")
print(A2)

# Centralidade de autovetor
eigenvalues, eigenvectors = np.linalg.eig(A)
principal_eigenvector = eigenvectors[:, 0]
print(f"\nCentralidade de autovetor: {principal_eigenvector}")

## 3. Calculando Propriedades de Rede com NetworkX

In [ ]:
import networkx as nx
import numpy as np

# Criar rede scale-free (Barabasi-Albert)
G = nx.barabasi_albert_graph(n=100, m=2)

# Analise
degrees = [G.degree(n) for n in G.nodes()]
print(f"Grau medio: {np.mean(degrees):.2f}")

# Clustering e Caminho Medio
C_global = nx.average_clustering(G)
print(f"Clustering global: {C_global:.3f}")

if nx.is_connected(G):
    L = nx.average_shortest_path_length(G)
    print(f"Caminho medio: {L:.3f}")

# Identificar hubs (nos com grau > 2*media)
threshold = 2 * np.mean(degrees)
hubs = [n for n in G.nodes() if G.degree(n) > threshold]
print(f"Hubs encontrados: {len(hubs)}")

## 4. Gerando Diferentes Modelos de Rede

In [ ]:
import networkx as nx

# 1. Rede Aleatoria (Erdos-Renyi)
G_random = nx.erdos_renyi_graph(n=50, p=0.1)

# 2. Rede Small-World (Watts-Strogatz)
G_sw = nx.watts_strogatz_graph(n=50, k=4, p=0.3)

# 3. Rede Scale-Free (Barabasi-Albert)
G_sf = nx.barabasi_albert_graph(n=50, m=2)

In [ ]:
# Comparar propriedades
models = [G_random, G_sw, G_sf]
names = ['Random', 'Small-World', 'Scale-Free']

for G, name in zip(models, names):
    C = nx.average_clustering(G)
    L = nx.average_shortest_path_length(G) if nx.is_connected(G) else float('inf')
    degrees = [d for n, d in G.degree()]

    print(f"\n{name}: Clustering: {C:.3f}, Path: {L:.3f}")

## 5. Acessando Bases de Dados com Python

In [ ]:
import requests
import networkx as nx

# Exemplo: Rede PPI do STRING
species, gene, thresh = '9606', 'TP53', 700
url = f'https://string-db.org/api/json/network?identifiers={gene}&species={species}&required_score={thresh}'
response = requests.get(url)
data = response.json()

In [ ]:
# Construir grafo
G = nx.Graph()
for interaction in data:
    n1, n2 = interaction['preferredName_A'], interaction['preferredName_B']
    G.add_edge(n1, n2, weight=interaction['score'])

print(f"Rede {gene}: {G.number_of_nodes()} nos, {G.number_of_edges()} arestas")

## 6. Simulando MCA (Metabolic Control Analysis) com Python

In [ ]:
import numpy as np
from scipy.optimize import fsolve

# Taxas
def v1(A, Vmax1=10, Km1=5): return Vmax1 * A / (Km1 + A)
def v2(B, Vmax2=8, Km2=4): return Vmax2 * B / (Km2 + B)

# Estado estacionario (v1 = v2)
A_fixed = 10
B_ss = fsolve(lambda B: v1(A_fixed) - v2(B), 5)[0]
J_ss = v2(B_ss)

print(f"B_ss = {B_ss:.2f}, Fluxo J = {J_ss:.2f}")

In [ ]:
def calc_C(idx, delta=0.01):
    def get_J(v1_v, v2_v):
        B = fsolve(lambda B: v1_v(A_fixed) - v2_v(B), 5)[0]
        return v2_v(B)
    
    if idx == 1:
        Jb = get_J(lambda a: v1(a, 10), v2)
        Jp = get_J(lambda a: v1(a, 10*(1+delta)), v2)
    else:
        Jb = get_J(v1, lambda b: v2(b, 8))
        Jp = get_J(v1, lambda b: v2(b, 8*(1+delta)))
    return ((Jp - Jb) / Jb) / delta

print(f"C1^J = {calc_C(1):.3f}, C2^J = {calc_C(2):.3f}")